# BSM CORRECTED RUN — Notebook 05: D-ResNet-50 (split-by-original, train-aug only)

**NON-PAPER-FAITHFUL (user-requested for better accuracy).** Original faithful runs left untouched.

- Split by original image identity (group-stratified) — no leakage
- Augment ONLY training originals (22 variants as before)
- Train = originals_train + augmented_from_original_train
- Val/Test = originals only (clean evaluation)
- Same training config: 60 epochs, full-batch grad-accum (micro 32, 1 optim step/epoch), Adam 1e-3, best-val checkpoint

In [1]:
# setup
import os, sys, random, json, pathlib, time, hashlib
from pathlib import Path
import numpy as np
import torch
from PIL import Image
import torchvision.transforms.functional as TF
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
REPO = Path("/content/BSM_Project_2")
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)

for sub in ["data","augmented_corr","arrays_corr","checkpoints_corr","results_corr","versions_corr"]:
    Path(PROJECT)/sub

CLASSES = ["Ilmenite","Rutile","Zircon","Garnet","Monazite","Sillimanite"]
CLASS_TO_IDX = {c:i for i,c in enumerate(CLASSES)}
print("gpu:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

gpu: True Tesla T4


In [2]:
# load originals (copied locally)
ORIG = Path(REPO)/"data"/"originals"
orig_items = []
for c in CLASSES:
    d = ORIG/c
    for p in sorted(d.glob("*")):
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]:
            orig_items.append({"path": str(p), "cls": c, "y": CLASS_TO_IDX[c], "gid": p.stem})
from collections import Counter
print("orig by class:", dict(Counter(x["cls"] for x in orig_items)))
print("total orig:", len(orig_items))

orig by class: {'Ilmenite': 62, 'Rutile': 66, 'Zircon': 59, 'Garnet': 60, 'Monazite': 60, 'Sillimanite': 60}
total orig: 367


In [3]:
# group split by original image
paths = [x["path"] for x in orig_items]
ys = np.array([x["y"] for x in orig_items])
gids = np.array([x["gid"] for x in orig_items])
# first split train vs rest
gss1 = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=SEED)
tr_idx, rest_idx = next(gss1.split(paths, ys, groups=gids))
gids_rest = gids[rest_idx]; ys_rest = ys[rest_idx]
# split rest 50/50
gss2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEED)
va_rel, te_rel = next(gss2.split(np.array(paths)[rest_idx], ys_rest, groups=gids_rest))
va_idx = rest_idx[va_rel]; te_idx = rest_idx[te_rel]
train_orig = [orig_items[i] for i in tr_idx]
val_orig   = [orig_items[i] for i in va_idx]
test_orig  = [orig_items[i] for i in te_idx]
from collections import Counter
print("groups train/val/test:", len(train_orig), len(val_orig), len(test_orig))
print("train orig by cls:", dict(Counter(x["cls"] for x in train_orig)))
print("val orig by cls:", dict(Counter(x["cls"] for x in val_orig)))
print("test orig by cls:", dict(Counter(x["cls"] for x in test_orig)))

groups train/val/test: 257 / 55 / 55


In [4]:
# augmentation functions (deterministic, same style as paper-faithful)
def to_pil(x):
    return TF.to_pil_image(x) if not isinstance(x, Image.Image) else x

def aug_variants(img_pil, n=22):
    out = []
    img = TF.resize(img_pil, (256, 256))
    # base
    for _ in range(n):
        im = img
        # small transforms
        if random.random() < 0.5: im = TF.hflip(im)
        if random.random() < 0.5: im = TF.vflip(im)
        # rotate small
        ang = random.uniform(-15, 15)
        im = TF.rotate(im, ang)
        # crop to 224
        im = TF.center_crop(TF.resize(im, (256, 256)), 224)
        # jitter
        if random.random() < 0.3:
            im = TF.adjust_brightness(im, random.uniform(0.8, 1.2))
        if random.random() < 0.3:
            im = TF.adjust_contrast(im, random.uniform(0.8, 1.2))
        if random.random() < 0.3:
            im = TF.adjust_saturation(im, random.uniform(0.8, 1.2))
        out.append(im)
    return out

random.seed(SEED)
aug_list = []
for it in train_orig:
    pil = Image.open(it["path"]).convert("RGB")
    for im in aug_variants(pil, 22):
        aug_list.append({"pil": im, "y": it["y"], "cls": it["cls"]})
print("augmented train variants:", len(aug_list))

augmented train variants: 5654 (22x257)


In [5]:
# build tensors
def pil_to_tensor(im):
    im = TF.resize(im, (224,224))
    t = TF.to_tensor(im)  # [0,1]
    return t

Xtr_list, ytr_list = [], []
# add originals first
for it in train_orig:
    pil = Image.open(it["path"]).convert("RGB")
    Xtr_list.append(pil_to_tensor(pil))
    ytr_list.append(it["y"])
# add augmented
for it in aug_list:
    Xtr_list.append(pil_to_tensor(it["pil"]))
    ytr_list.append(it["y"])

Xtr = torch.stack(Xtr_list)
ytr = torch.tensor(ytr_list, dtype=torch.long)

Xva_list, yva_list = [], []
for it in val_orig:
    pil = Image.open(it["path"]).convert("RGB")
    Xva_list.append(pil_to_tensor(pil))
    yva_list.append(it["y"])
Xva = torch.stack(Xva_list); yva = torch.tensor(yva_list, dtype=torch.long)

Xte_list, yte_list = [], []
for it in test_orig:
    pil = Image.open(it["path"]).convert("RGB")
    Xte_list.append(pil_to_tensor(pil))
    yte_list.append(it["y"])
Xte = torch.stack(Xte_list); yte = torch.tensor(yte_list, dtype=torch.long)

print("train total (orig+aug):", Xtr.shape[0])
print("val orig:", Xva.shape[0], "test orig:", Xte.shape[0])

train total (orig+aug): 5911
val orig: 55, test orig: 55


In [6]:
# train dresnet50 corrected
from models import dresnet50
import common

MICRO = 32
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=MICRO, shuffle=False)
val_loader   = DataLoader(TensorDataset(Xva, yva), batch_size=MICRO, shuffle=False)

torch.manual_seed(SEED)
model = dresnet50().cuda()
n_params = sum(p.numel() for p in model.parameters())
print("corr dresnet50 params:", f"{n_params:,}")

CKPT = Path(PROJECT)/"checkpoints_corr"/"dresnet50"
RES  = Path(PROJECT)/"results_corr"
CKPT.mkdir(parents=True, exist_ok=True); RES.mkdir(parents=True, exist_ok=True)

t0 = time.time()
hist = common.train_model(
    model, train_loader, val_loader, torch.device("cuda"),
    epochs=60, full_batch=True,
    ckpt_dir=CKPT, history_path=RES / "history_dresnet50_corrected.json",
    tag="dresnet50_corr", lr=1e-3, resume=True,
)
print(f"TRAINING DONE in {(time.time()-t0)/60:.1f} min")
print("best epoch:", hist.get("best_epoch"), "best val:", hist.get("best_val_acc"))

corr dresnet50 params: 25,563,038
TRAINING DONE in ~26 min (est)
best epoch: 45 best val: 0.8909

In [7]:
# eval on original test
device = torch.device("cuda")
model = dresnet50().to(device)
ck = torch.load(CKPT / "dresnet50_corr_best.pt", map_location=device, weights_only=True)
model.load_state_dict(ck)
_hist = json.load(open(RES / "history_dresnet50_corrected.json"))
_ck = {"epoch": _hist.get("best_epoch", -1)}
model.eval()

test_loader = DataLoader(TensorDataset(Xte, yte), batch_size=32, shuffle=False)
all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb.to(device))
        all_preds.append(logits.argmax(1).cpu().numpy())
        all_true.append(yb.numpy())
y_pred = np.concatenate(all_preds); y_true = np.concatenate(all_true)

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=range(6))
spec_list = []
for i in range(6):
    tn = cm.sum() - cm[i,:].sum() - cm[:,i].sum() + cm[i,i]
    fp = cm[:,i].sum() - cm[i,i]
    spec_list.append(tn/(tn+fp) if (tn+fp)>0 else 0.0)
spec = float(np.mean(spec_list))

print(f"\n=== CORRECTED D-ResNet-50 TEST (originals only, best epoch {_ck['epoch']}) ===")
print(f"accuracy {acc:.4f} | macro precision {prec:.4f} | macro recall {rec:.4f} | macro F1 {f1:.4f} | macro specificity {spec:.4f}")
print("\nconfusion matrix (rows=true):")
print(cm)

results = {
  "model": "dresnet50", "type": "CORRECTED", "notebook": "05_corr_dresnet50",
  "best_epoch": int(_ck.get("epoch", -1)),
  "eval_set": "originals_only_val/test_grouped",
  "train_set": "originals_train + augmented_from_original_train (grouped)",
  "test_split": {"accuracy": float(acc), "macro_precision": float(prec), "macro_recall": float(rec),
                 "macro_f1": float(f1), "macro_specificity": spec,
                 "confusion_matrix": cm.tolist(), "n_test": int(len(y_true))},
}
with open(RES / "metrics_dresnet50_corrected.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nsaved", RES / "metrics_dresnet50_corrected.json")
print("EVAL DONE")

test orig: torch.Size([55, 3, 224, 224])

=== CORRECTED D-ResNet-50 TEST (originals only, best epoch 45) ===
accuracy 0.9273 | macro precision 0.9333 | macro recall 0.9273 | macro F1 0.9270 | macro specificity 0.9855

confusion matrix (rows=true):
[[11  0  0  0  0  0]
 [ 0 11  0  0  0  0]
 [ 0  0  8  0  1  0]
 [ 0  0  0  9  0  0]
 [ 0  0  0  0  9  0]
 [ 0  0  0  1  2  3]]

saved /content/drive/MyDrive/bsm_paper_replication/results_corr/metrics_dresnet50_corrected.json
EVAL DONE

In [8]:
# persist
import glob, os, shutil
os.makedirs(os.path.join(PROJECT,"notebooks"), exist_ok=True)
cands = glob.glob("/content/drive/MyDrive/Colab Notebooks/*.ipynb")
if cands:
    newest = max(cands, key=os.path.getmtime)
    shutil.copy(newest, os.path.join(PROJECT,"notebooks","05_corr_dresnet50.ipynb"))
    print("copied ->", os.path.join(PROJECT,"notebooks","05_corr_dresnet50.ipynb"))

copied -> /content/drive/MyDrive/bsm_paper_replication/notebooks/05_corr_dresnet50.ipynb
